<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/lab/2026-10-16-Walmart-Warm-Up-SKU-Mess-Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [5]:
import pandas as pd
PATH = 'https://raw.githubusercontent.com/jasontpv/DS2002FA26/main/notebooks/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [6]:
# TODO (hint: .str.lower().str.contains('pop'))
pop = cat[cat['product_name'].str.lower().str.contains('pop')]
print('Pop-Tart rows:', len(pop))
print(pop[['sku', 'product_name', 'category']])

Pop-Tart rows: 4
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [7]:
# TODO
straw = pop[pop['product_name'].str.lower().str.contains('strawberry')]
print(straw[['sku', 'product_name']])

not_straw = pop[~pop['product_name'].str.lower().str.contains('strawberry')]
print('Pop-Tart rows that are not strawberry:', len(not_straw))

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry
Pop-Tart rows that are not strawberry: 0


**Pop-Tart rows that are a different product:** There aren't any in this catalog, all 4 are strawberry. If there were another flavor (like the blueberry one in Wednesday's studio), it would stay separate because a different flavor is a different product.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [8]:
# TODO
CANON = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

rows_before = len(cat)
cat['canonical_sku'] = cat['sku'].replace(CANON)
print(cat.loc[cat['canonical_sku'] == 'POPTART-STRAW', ['sku', 'canonical_sku']])
print('NaNs if I had used .map instead:', cat['sku'].map(CANON).isna().sum())

             sku  canonical_sku
0          PT-12  POPTART-STRAW
1     SKU#459812  POPTART-STRAW
2  POPTART-STRAW  POPTART-STRAW
3  PT_STRAWBERRY  POPTART-STRAW
NaNs if I had used .map instead: 42


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [9]:
# TODO
print('BEFORE:')
print(cat['category'].value_counts())

cat['category_clean'] = cat['category'].str.strip().str.title()
print('\nAFTER:')
print(cat['category_clean'].value_counts())

BEFORE:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

AFTER:
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast

Before cleaning there were 27 different category labels, mostly from capitalization (EMERGENCY, Emergency, emergency). After strip and title-case there are 15. Some pairs are still two labels for the same thing: Emergency and Emergency Supplies, Grocery and Groceries, Breakfast and Breakfast Foods, Alcohol and Alcoholic Beverages, and Baby and Baby Products.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [10]:
# TODO
cat['name_key'] = (cat['product_name'].str.lower()
                   .str.replace(r'[^a-z0-9 ]', ' ', regex=True)
                   .str.split()
                   .apply(sorted)
                   .str.join(' '))

groups = cat.groupby('name_key')['sku'].agg(['count', list])
candidates = groups[groups['count'] > 1]
print(candidates)
print(len(candidates), 'name groups with more than one SKU')

                      count                                   list
name_key                                                          
24 beer light pack        2                    [BEER-24, LTBEER24]
4 aa batteries pack       2                 [BAT-AA4, BATT_AA_4PK]
aid first kit             2                [FAID-01, FIRSTAID-KIT]
board game monopoly       3      [BG-MONO, MONOPOLY-01, GAME-MONO]
bread white               2                 [BREAD-WH, WHITEBREAD]
butter peanut             2                    [PB-16, PNUTBTR-16]
candles emergency         2                 [CNDL-EM, CANDLE-EMRG]
canned soup               2                  [SOUP-CAN, CNSOUP-01]
cards playing             2               [CARDS-01, PLAYCARD-STD]
chips pack variety        2                [CHIPS-VP, CHIPVARIETY]
duty heavy tarp           2                   [TARP-HD, HDTARP-01]
flashlight led            2                  [FLASH-01, LED-FLASH]
paper towels              2                [PTOWEL-6, PAPERTOW

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [11]:
# TODO
assert cat['canonical_sku'].notna().all(), 'mapping created nulls'
assert len(cat) == rows_before, 'mapping changed the row count'
print('mapping is safe:', len(cat), 'rows,', cat['canonical_sku'].nunique(), 'canonical SKUs')

mapping is safe: 45 rows, 42 canonical SKUs


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Bottled water and AA batteries are two other product families that look fragmented. Water has 3 SKUs (WTR-24, WATER24PK, WTR_24PK) for “Bottled Water 24PK”, “Bottled Water 24-Pack” and “Water - 24 Pack”. AA batteries has 3 (BATT-100, BAT-AA4, BATT_AA_4PK). I'd group each one the same way as the Pop-Tarts, by mapping every variant to only one SKU (like WATER-24 and BATT-AA-4).

b) It becomes a business decision when the data can't tell you if two products are actually the same thing. For example, STREAM-01 is “Roku Streaming Stick” and ROKU-STD is just “Streaming Device.” They're priced about the same and the SKU says Roku, but “Streaming Device” could be a different brand. If I merge them and I'm wrong, Roku sales look bigger than they are. I'd ask someone on the merchandising or inventory team who manages the catalog, because they'd know what's actually being sold.